In [1]:
import pandas as pd
import numpy as np
import json
from tqdm import tqdm


In [2]:
df = pd.read_parquet("../logs-2025-12-01.parquet")

df.head()


,log_id,game_session_id,player_id,played_by,board_state,legal_moves,chosen_move,game_status,logged_at,ai_difficulty,...,move_confidence,branching_factor,explored_actions,mcts_tree_size,rollout_count,rollout_depth_average,rollout_depth_max,mcts_search_duration_ms,ai_latency_ms,outcome
0,b845c355-6965-4192-87f6-a227440d5edf,15a3eb8f-28e2-4aea-b4e3-0dc58b933252,bf7308cc-0199-4c2b-badf-82c819fc581b,O,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 1, ""row"": 0, ""player"": {""mark"": ""O"", ...","{""col"": 0, ""row"": 2, ""bestQValue"": 0.917368421...",IN_PROGRESS,2025-12-01T11:48:08.991819,MEDIUM,...,0.950,6,6,172,1000,0.227,4,15,15,NONE
1,e4148ce5-3435-4b73-a604-0435690a410e,c57cfb6d-f6cb-4cb6-80f3-8351ea9f7b7b,559b50b1-be4e-42de-8bd0-20f422541381,X,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 1, ""row"": 0, ""player"": {""mark"": ""X"", ...","{""col"": 0, ""row"": 2, ""bestQValue"": 0.872990353...",IN_PROGRESS,2025-12-01T11:48:10.079332,MEDIUM,...,0.311,7,7,800,1000,1.510,6,60,60,NONE
2,03d9d5e9-3c15-4d66-b050-a47591c8026d,90431947-b4bd-44dc-9957-7d2ebf9e7f12,f87b4fb6-87fc-4c0b-bf57-dd6314b4b07f,O,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 1, ""row"": 0, ""player"": {""mark"": ""O"", ...","{""col"": 2, ""row"": 0, ""bestQValue"": 0.921706263...",IN_PROGRESS,2025-12-01T11:48:10.688832,MEDIUM,...,0.926,6,6,166,1000,0.235,5,23,23,NONE
3,e232c04a-7e15-40e4-a547-269cf7bfd03c,94333c80-6b8c-49f5-b3ac-ca908db69e28,88f47797-0200-4779-8b47-18a3e34b4861,X,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 0, ""row"": 0, ""player"": {""mark"": ""X"", ...","{""col"": 2, ""row"": 2, ""bestQValue"": 0.872340425...",IN_PROGRESS,2025-12-01T11:48:11.547821,MEDIUM,...,0.282,7,7,800,1000,1.506,6,44,44,NONE
4,d979d574-040d-4718-8e94-5747bb927d34,58f459b5-15e9-484c-8e01-59347cd5456e,df80b476-53bb-4c8b-96e1-a07915ad6e52,O,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 0, ""row"": 0, ""player"": {""mark"": ""O"", ...","{""col"": 0, ""row"": 0, ""bestQValue"": 0.397142857...",IN_PROGRESS,2025-12-01T11:48:09.400081,MEDIUM,...,0.350,8,8,997,1000,2.679,7,90,90,NONE


In [3]:
def board_to_json(board_array):
    board_list = []
    for r in range(3):
        for c in range(3):
            value = board_array[r, c]  # "EMPTY", "X", or "O"
            board_list.append({"col": c, "row": r, "value": value})
    return json.dumps({"board": board_list})

def parse_chosen_move(s):
    obj = json.loads(s)
    return obj["row"], obj["col"]


In [4]:
fixed_board_states = []

for game_id, group in tqdm(df.groupby("game_session_id"), desc="Reconstructing boards"):
    # sort by time
    group = group.sort_values("logged_at")

    # start with empty board
    board = np.full((3, 3), "EMPTY", dtype=object)

    # replay each move
    for idx, row in group.iterrows():
        move_row, move_col = parse_chosen_move(row["chosen_move"])
        mark = row["played_by"]  # "X" or "O"

        # place mark
        board[move_row, move_col] = mark

        # convert board to JSON as a string
        board_json = board_to_json(board)

        fixed_board_states.append((idx, board_json))


Reconstructing boards: 100%|██████████| 510/510 [00:00<00:00, 1074.08it/s]


In [5]:
fixed_board_df = pd.DataFrame(fixed_board_states, columns=["index", "board_state_fixed"])
fixed_board_df = fixed_board_df.set_index("index")

df["board_state_fixed"] = fixed_board_df["board_state_fixed"]

df.head()


,log_id,game_session_id,player_id,played_by,board_state,legal_moves,chosen_move,game_status,logged_at,ai_difficulty,...,branching_factor,explored_actions,mcts_tree_size,rollout_count,rollout_depth_average,rollout_depth_max,mcts_search_duration_ms,ai_latency_ms,outcome,board_state_fixed
0,b845c355-6965-4192-87f6-a227440d5edf,15a3eb8f-28e2-4aea-b4e3-0dc58b933252,bf7308cc-0199-4c2b-badf-82c819fc581b,O,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 1, ""row"": 0, ""player"": {""mark"": ""O"", ...","{""col"": 0, ""row"": 2, ""bestQValue"": 0.917368421...",IN_PROGRESS,2025-12-01T11:48:08.991819,MEDIUM,...,6,6,172,1000,0.227,4,15,15,NONE,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""O""},..."
1,e4148ce5-3435-4b73-a604-0435690a410e,c57cfb6d-f6cb-4cb6-80f3-8351ea9f7b7b,559b50b1-be4e-42de-8bd0-20f422541381,X,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 1, ""row"": 0, ""player"": {""mark"": ""X"", ...","{""col"": 0, ""row"": 2, ""bestQValue"": 0.872990353...",IN_PROGRESS,2025-12-01T11:48:10.079332,MEDIUM,...,7,7,800,1000,1.510,6,60,60,NONE,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""O""},..."
2,03d9d5e9-3c15-4d66-b050-a47591c8026d,90431947-b4bd-44dc-9957-7d2ebf9e7f12,f87b4fb6-87fc-4c0b-bf57-dd6314b4b07f,O,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 1, ""row"": 0, ""player"": {""mark"": ""O"", ...","{""col"": 2, ""row"": 0, ""bestQValue"": 0.921706263...",IN_PROGRESS,2025-12-01T11:48:10.688832,MEDIUM,...,6,6,166,1000,0.235,5,23,23,NONE,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""O""},..."
3,e232c04a-7e15-40e4-a547-269cf7bfd03c,94333c80-6b8c-49f5-b3ac-ca908db69e28,88f47797-0200-4779-8b47-18a3e34b4861,X,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 0, ""row"": 0, ""player"": {""mark"": ""X"", ...","{""col"": 2, ""row"": 2, ""bestQValue"": 0.872340425...",IN_PROGRESS,2025-12-01T11:48:11.547821,MEDIUM,...,7,7,800,1000,1.506,6,44,44,NONE,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT..."
4,d979d574-040d-4718-8e94-5747bb927d34,58f459b5-15e9-484c-8e01-59347cd5456e,df80b476-53bb-4c8b-96e1-a07915ad6e52,O,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""EMPT...","[{""col"": 0, ""row"": 0, ""player"": {""mark"": ""O"", ...","{""col"": 0, ""row"": 0, ""bestQValue"": 0.397142857...",IN_PROGRESS,2025-12-01T11:48:09.400081,MEDIUM,...,8,8,997,1000,2.679,7,90,90,NONE,"{""board"": [{""col"": 0, ""row"": 0, ""value"": ""O""},..."


In [7]:
import json
import numpy as np

def show_board(board):
    """Pretty print a 3x3 TicTacToe board."""
    for r in range(3):
        print(board[r])
    print("-" * 15)


def replay_game(df, game_session_id):
    # Filter game
    game = df[df["game_session_id"] == game_session_id].copy()
    game = game.sort_values("logged_at")

    # Start empty board
    board = np.full((3, 3), "EMPTY", dtype=object)

    print(f"Replaying game: {game_session_id}")
    print("=" * 30)

    for move_number, (_, row) in enumerate(game.iterrows(), start=1):
        # Parse move
        move = json.loads(row["chosen_move"])
        r, c = move["row"], move["col"]
        mark = row["played_by"]

        # Apply move to board
        board[r, c] = mark

        # Print move info
        print(f"Move {move_number}: Player {mark} placed at (row={r}, col={c})")
        show_board(board)


# ---- Usage ----
# Replace with any real session ID from your dataset
sample_session = df["game_session_id"].iloc[0]
replay_game(df, sample_session)


Replaying game: 15a3eb8f-28e2-4aea-b4e3-0dc58b933252
Move 1: Player X placed at (row=1, col=1)
['EMPTY' 'EMPTY' 'EMPTY']
['EMPTY' 'X' 'EMPTY']
['EMPTY' 'EMPTY' 'EMPTY']
---------------
Move 2: Player O placed at (row=0, col=0)
['O' 'EMPTY' 'EMPTY']
['EMPTY' 'X' 'EMPTY']
['EMPTY' 'EMPTY' 'EMPTY']
---------------
Move 3: Player X placed at (row=0, col=2)
['O' 'EMPTY' 'X']
['EMPTY' 'X' 'EMPTY']
['EMPTY' 'EMPTY' 'EMPTY']
---------------
Move 4: Player O placed at (row=2, col=0)
['O' 'EMPTY' 'X']
['EMPTY' 'X' 'EMPTY']
['O' 'EMPTY' 'EMPTY']
---------------
Move 5: Player X placed at (row=1, col=2)
['O' 'EMPTY' 'X']
['EMPTY' 'X' 'X']
['O' 'EMPTY' 'EMPTY']
---------------
Move 6: Player O placed at (row=1, col=0)
['O' 'EMPTY' 'X']
['O' 'X' 'X']
['O' 'EMPTY' 'EMPTY']
---------------


In [8]:
df.to_parquet("../logs_fixed_board_state.parquet")


In [9]:
# Test parsing a single chosen_move
test = df.iloc[0]
print(test["chosen_move"])

parsed = json.loads(test["chosen_move"])
print("Parsed:", parsed)
print("Move row:", parsed["row"], "col:", parsed["col"])


{"col": 0, "row": 2, "bestQValue": 0.9173684210526316, "aiLatencyMs": 15, "aiDifficulty": "MEDIUM", "mctsTreeSize": 172, "rolloutCount": 1000, "bestVisitCount": 950, "mctsIterations": 1000, "moveConfidence": 0.95, "branchingFactor": 6, "exploredActions": 6, "rolloutDepthMax": 4, "visitCountsJson": "{0,1=10, 1,0=9, 1,2=8, 2,0=950, 2,1=14, 2,2=9}", "actionScoresJson": "{0,1=0.15, 1,0=0.1111111111111111, 1,2=0.0625, 2,0=0.9173684210526316, 2,1=0.2857142857142857, 2,2=0.1111111111111111}", "policyVectorJson": "{0,1=0.01, 1,0=0.009, 1,2=0.008, 2,0=0.95, 2,1=0.014, 2,2=0.009}", "explorationConstant": 0.7071067811865475, "rolloutDepthAverage": 0.227, "mctsSearchDurationMs": 15}
Parsed: {'col': 0, 'row': 2, 'bestQValue': 0.9173684210526316, 'aiLatencyMs': 15, 'aiDifficulty': 'MEDIUM', 'mctsTreeSize': 172, 'rolloutCount': 1000, 'bestVisitCount': 950, 'mctsIterations': 1000, 'moveConfidence': 0.95, 'branchingFactor': 6, 'exploredActions': 6, 'rolloutDepthMax': 4, 'visitCountsJson': '{0,1=10, 1,0